In [1]:
import glob
import pandas as pd
from pathlib import Path
from typing import Callable

files = glob.glob("/g/data/ct11/access-nri/replicas/esmvaltool/**/*.nc",recursive=True)
print(f"No. of obs6 files: {pd.Series(files).str.lower().str.contains('obs6').sum()}")
srs = pd.Series(files)
srs.name='path'

for f in files[:10]:
    print(f)

No. of obs6 files: 2886
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Emon_tdps_201701-201712.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_day_rsds_201101-201112.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_hur_199401-199412.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_rsutcs_199401-199412.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_vas_200801-200812.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_rsut_198101-198112.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_tauu_199201-199212.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interi

In [2]:
def get_tier(file_path : str) -> str:
    """ Split the parts, find the one that is 
    either Tier1, Tier2, or Tier3.
    """
    path_parts = Path(file_path).parts
    for part in path_parts:
        if part in ['Tier1', 'Tier2', 'Tier3']:
            return part
    return None

def filter_for_str(srs: pd.Series, filter_str: str) -> pd.Series:
    """ Apply filtering function to each file path and expand results into separate columns """
    if not isinstance(srs, pd.Series):
        raise TypeError(f'Wrong dtype, probably the wrong input source. Expected {type(srs)}=pd.Series, got {type(srs)=}')
    
    return srs[
        srs.str.lower().str.contains(filter_str)
    ].reset_index(drop=True)

def split(filtered_srs: pd.Series, split_func: Callable) -> pd.DataFrame:
    """
    Take our filtered series & split out the various identifiers in the file stems using
    the `split_func` we pass in.
    """
    COLNAMES = ['project_id', 'source_id', 'experiment_id', 'version',  'table_id', 'variable_id', 'time_range', 'tier']
    df = pd.DataFrame(filtered_srs)
    df[['project_id', 'source_id', 'experiment_id', 'version', 'table_id', 'variable_id', 'time_range', 'tier']] = df['path'].apply(lambda x: pd.Series(split_func(x)))
    return df
            

In [3]:
from pathlib import Path
from typing import Callable

# Function to split the filename into components based on '_'
def split_filename_obs6(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range, *_]:
            return {
                'project_id': project_id,
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case [project_id, source_id, experiment_id, version, table_id, variable_id]:
            return {
                'project_id': project_id,
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': None,
                'tier' : get_tier(file_path),
            }  
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )
            


obs6_srs = filter_for_str(srs,'obs6')
obs6_df = split(obs6_srs, split_filename_obs6)


In [4]:
df_all = pd.DataFrame(pd.Series(files))
df_all.columns = ['path']
df_remaining = df_all[~pd.Series(files).str.lower().str.contains('obs6')].copy().reset_index(drop=True)

obs4mips_srs = filter_for_str(srs,'obs4mips')
obs4mips_df = pd.DataFrame(obs4mips_srs)


def split_filename_obs4mips(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    
    if "-" in components[-1]:
        time_range = components[-1]
    else:
        time_range = "-".join([components[-2], components[-1]])
    return {
        'project_id': "obs4MIPs",
        'source_id': components[2],
        'version': components[4],
        'variable_id': components[0],
        'time_range': time_range,
        'tier' : get_tier(file_path),
        } 
    
# Apply function to each file path and expand results into separate columns

obs4mips_df[['project_id', 'source_id', 'version', 'variable_id', 'time_range','tier']] = obs4mips_df['path'].apply(lambda x: pd.Series(split_filename_obs4mips(x)))
obs4mips_df

,path,project_id,source_id,version,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,cllcalipso,20071201-20071231,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,clhcalipso,20100501-20100531,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,clmcalipso,20080801-20080831,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,clccalipso,20080701-20080731,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,cllcalipso,20090501-20090531,Tier1
...,...,...,...,...,...,...,...
1741,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,albisccp,200402-200402,Tier1
1742,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,albisccp,199504-199504,Tier1
1743,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,albisccp,200001-200001,Tier1
1744,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,cltisccp,198812-198812,Tier1


In [5]:
def split_filename_CFSR(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `zg_Amon_reanalysis_CFSR_201301-201312` which get mapped as
    f{variable_id}_{table_id}_{experiment_id}_{source_id}_{time_range}
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, table_id, experiment_id, source_id, time_range]:
            return {
                'project_id': "obs4MIPs",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': None,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )


cfsr_srs = filter_for_str(srs,'cfsr')
cfsr_df = split(cfsr_srs, split_filename_CFSR)

cfsr_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,201301-201312,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,va,200601-200612,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,198101-198112,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,200701-200712,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,198501-198512,Tier1
...,...,...,...,...,...,...,...,...,...
165,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,ua,201101-201112,Tier1
166,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,200201-200212,Tier1
167,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,ua,199601-199612,Tier1
168,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,ua,199901-199912,Tier1


In [6]:
def split_filename_CERES_EBAF(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `rlds_CERES-EBAF_L3B_Ed2-7_200003-201209` which get mapped as
    f{variable_id}_{source_id}_{experiment_id}_{version_}_{time_range}
    or 
    `OBS_CERES-EBAF_sat_Ed4.0_Amon_rsut_200003-201812` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"

    
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id, experiment_id, version, time_range]:
            return {
                'project_id': "obs4MIPs",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': None,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                'project_id': project_id,
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )


ceres_ebaf_srs = filter_for_str(srs,'ceres-ebaf')
ceres_ebaf_df = split(ceres_ebaf_srs, split_filename_CERES_EBAF)

ceres_ebaf_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rsut,200003-201812,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rlutcs,200003-201812,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rlut,200003-202203,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rsut,200003-202203,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rsutcs,200003-202203,Tier2
5,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rlutcs,200003-202203,Tier2
6,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rsutcs,200003-201812,Tier2
7,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rlut,200003-201812,Tier2
8,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CERES-EBAF,L3B,Ed2-8,None,rsdscs,200003-201311,Tier1
9,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CERES-EBAF,L3B,Ed2-8,None,rlus,200003-201311,Tier1


In [7]:
# 
def split_filename_JRA55(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `rsut_Amon_reanalysis_JRA-55_195801-201912` which get mapped as
    f{variable_id}_{table_id}_{experiment_id}_{source_id}_{time_range}
    or 
    `OBS6_JRA-55_reanaly_1_Amon_tas_195801-202212` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"

    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, table_id, experiment_id, source_id, time_range]:
            return {
                'project_id': "obs4MIPs",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': None,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case [_, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                'project_id': "OBS6",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

jra55_srs = filter_for_str(srs,'jra-55')
jra55_df = split(jra55_srs, split_filename_JRA55)


jra55_flist = jra55_df['path'].tolist()
obs6_flist = obs6_df['path'].tolist()

### For whatever reason about half of these overlap? IDK, We drop the duplicates later on 
print(len(set(jra55_flist).intersection(set(obs6_flist))))
print(len(jra55_df))

16
29


In [8]:
#  '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/ESACCI-SST/OBS_ESACCI-SST_sat_2.2_Amon_ts_201901-201912.nc',
# '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/ESACCI-SST/OBS_ESACCI-SST_sat_L4-GHRSST-SSTdepth-OSTIA-GLOB_Amon_tsStderr_199201-199201.nc'

def split_filename_ESACCI(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_ESACCI-SST_sat_2.2_Amon_ts_201901-201912` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
    or 
    `OBS_ESACCI-SST_sat_L4-GHRSST-SSTdepth-OSTIA-GLOB_Amon_tsStderr_199201-199201` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{grid_label}_{table_id}_{variable_id}_{time_range}"
    or
    `xco2_ghgcci_l3_v100_200301_201412.nc`, which get mapped as
    f"{variable_id`_{source_id}_{grid_label}_{version}_{t1}_{t2}" => f"{variable_id`_{}_{}_{version}_{time_range}"
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [variable_id, source_id, version, _, t_start, t_end]:
            return {
                'project_id': "obs4MIPs",
                'source_id': source_id,
                'experiment_id': None,
                'version': version,
                'table_id': None,
                'variable_id': variable_id,
                'time_range': f"{t_start}-{t_end}",
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

esacci_srs = filter_for_str(srs,'esacci')
esacci_df = split(esacci_srs, split_filename_ESACCI)

esacci_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,sm,197901-201612,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,smStderr,197901-201612,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,dosStderr,197901-201612,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,dos,197901-201612,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,ESACCI-OC,sat,fv5.0,Omon,chl,199709-202012,Tier2
...,...,...,...,...,...,...,...,...,...
595,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-OZONE,sat,L3,Amon,tozStderr,199701-201012,Tier2
596,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-OZONE,sat,L3,Amon,toz,199701-201012,Tier2
597,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-OZONE,sat,L3,Amon,tro3prof,200701-200812,Tier2
598,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ghgcci,None,l3,None,xco2,200301-201412,Tier1


In [9]:
# '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/ESRL/OBS_ESRL_ground_PAL_Amon_co2s_200112-201912.nc',

def split_filename_ESRL(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_ESRL_ground_PAL_Amon_co2s_200112-201912.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{_}_{table_id}_{variable_id}_{time_range}"
 
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, _, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': None,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

esrl_srs = filter_for_str(srs,'esrl')
esrl_df = split(esrl_srs, split_filename_ESRL)

esrl_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,197903-199008,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,201003-201912,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,199710-200908,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,198001-202008,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,199901-201105,Tier2
...,...,...,...,...,...,...,...,...,...
89,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,200204-201705,Tier2
90,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,197908-201912,Tier2
91,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,198612-201707,Tier2
92,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,199701-201912,Tier2


In [10]:
#  '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/NCEP/OBS_NCEP_reanaly_1_Amon_zg_201001-201012.nc',

def split_filename_NCEP(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_NCEP_reanaly_1_Amon_zg_201001-201012.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
 
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

ncep_srs = filter_for_str(srs,'ncep')
ncep_df = split(ncep_srs, split_filename_NCEP)

ncep_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,wap,197901-202409,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,hur,197901-202409,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,tauu,197901-202409,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,clt,197901-202409,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,prw,197901-202409,Tier2
...,...,...,...,...,...,...,...,...,...
1248,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,day,rlut,199501-199512,Tier2
1249,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,day,pr,198601-198612,Tier2
1250,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,Amon,zg,196501-196512,Tier2
1251,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,day,pr,195501-195512,Tier2


In [11]:
#  '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/WFDE5/OBS_WFDE5_ground_v1.1-CRU_day_tas_201301-201312.nc',

def split_filename_WFDE(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_WFDE5_ground_v1.1-CRU_day_tas_201301-201312.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
 
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

wfde_srs = filter_for_str(srs,'wfde')
wfde_df = split(wfde_srs, split_filename_WFDE)

wfde_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU+GPCC,day,pr,199401-199412,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,tas,199401-199412,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,tas,200401-200412,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU+GPCC,day,pr,199501-199512,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,pr,198001-198012,Tier2
...,...,...,...,...,...,...,...,...,...
227,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,tas,200501-200512,Tier2
228,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,day,tas,198601-198612,Tier2
229,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,pr,200401-200412,Tier2
230,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,pr,201301-201312,Tier2


In [12]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/OSI-450-nh/OBS_OSI-450-nh_reanaly_v2_OImon_sic_201501-201512.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/OSI-450-nh/OBS_OSI-450-nh_reanaly_v2_day_sic_197901-197912.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/OSI-450-nh/OBS_OSI-450-nh_reanaly_v2_fx_areacello.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/OSI-450-sh/OBS_OSI-450-sh_reanaly_v2_OImon_sic_197901-197912.nc

def split_filename_OSI450(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_OSI-450-nh_reanaly_v2_OImon_sic_201501-201512.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
    or 
    `OBS_OSI-450-nh_reanaly_v2_fx_areacello.nc` which get mapped as 
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}"
    
    
 
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [project_id, source_id, experiment_id, version, table_id, variable_id]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': None,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

osi450_srs = filter_for_str(srs,'osi-450')
osi450_df = split(osi450_srs, split_filename_OSI450)

osi450_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-nh,reanaly,v2,OImon,sic,198101-198112,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-nh,reanaly,v2,day,sic,199301-199312,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-nh,reanaly,v2,OImon,sic,198501-198512,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-nh,reanaly,v2,day,sic,200701-200712,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-nh,reanaly,v2,OImon,sic,200101-200112,Tier2
...,...,...,...,...,...,...,...,...,...
145,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-sh,reanaly,v2,OImon,sic,198001-198012,Tier2
146,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-sh,reanaly,v2,day,sic,199801-199812,Tier2
147,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-sh,reanaly,v2,OImon,sic,200401-200412,Tier2
148,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,OSI-450-sh,reanaly,v2,OImon,sic,199301-199312,Tier2


In [13]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/REGEN/OBS_REGEN_reanaly_V1-2019_day_pr_201601-201612.nc

def split_filename_REGEN(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_REGEN_reanaly_V1-2019_day_pr_201601-201612.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

regen_srs = filter_for_str(srs,'regen')
regen_df = split(regen_srs, split_filename_REGEN)

regen_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,day,pr,196701-196712,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,Amon,pr,197201-197212,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,Amon,pr,195801-195812,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,Amon,pr,197101-197112,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,Amon,pr,195001-195012,Tier2
...,...,...,...,...,...,...,...,...,...
129,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,Amon,pr,198501-198512,Tier2
130,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,Amon,pr,196901-196912,Tier2
131,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,day,pr,199501-199512,Tier2
132,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,REGEN,reanaly,V1-2019,Amon,pr,196601-196612,Tier2


In [14]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/WOA/OBS_WOA_clim_2013v2_Omon_tos_200007-200007.nc

def split_filename_WOA(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_WOA_clim_2013v2_Omon_tos_200007-200007.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

woa_srs = filter_for_str(srs,'woa')
woa_df = split(woa_srs, split_filename_WOA)

woa_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,WOA,clim,2018,Oyr,no3,200001-200012,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WOA,clim,2013v2,Oyr,po4,200001-200012,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,WOA,clim,2018,Oyr,si,200001-200012,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,WOA,clim,2018,Omon,so,200007-200007,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WOA,clim,2013v2,Omon,sos,200007-200007,Tier2
5,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WOA,clim,2013v2,Omon,tos,200007-200007,Tier2
6,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WOA,clim,2013v2,Omon,thetao,200001-200012,Tier2
7,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,WOA,clim,2018,Omon,tos,200007-200007,Tier2
8,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,WOA,clim,2018,Oyr,po4,200001-200012,Tier2
9,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WOA,clim,2013v2,Oyr,no3,200001-200012,Tier2


In [15]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/PIOMAS/OBS_PIOMAS_reanaly_2.1_day_sithick_201801-201812.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/PIOMAS/OBS_PIOMAS_reanaly_2.1_fx_areacello.nc

def split_filename_PIOMAS(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_PIOMAS_reanaly_2.1_day_sithick_201801-201812.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
    or
    `OBS_PIOMAS_reanaly_2.1_fx_areacello.nc`
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}"


    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [project_id, source_id, experiment_id, version, table_id, variable_id]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': None,
                    'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

piomas_srs = filter_for_str(srs,'piomas')
piomas_df = split(piomas_srs, split_filename_PIOMAS)

piomas_df.head(5)

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PIOMAS,reanaly,2.1,day,sithick,200801-200812,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PIOMAS,reanaly,2.1,day,sithick,199001-199012,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PIOMAS,reanaly,2.1,day,sithick,200701-200712,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PIOMAS,reanaly,2.1,day,sithick,199401-199412,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PIOMAS,reanaly,2.1,day,sithick,199701-199712,Tier2


In [16]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/PERSIANN-CDR/OBS_PERSIANN-CDR_reanaly_v01r01_day_pr_201601-201612.nc

def split_filename_PERSIANN(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_PERSIANN-CDR_reanaly_v01r01_day_pr_201601-201612.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"


    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        # case [project_id, source_id, experiment_id, version, table_id, variable_id]:
        #     return {
        #             'project_id': project_id,
        #             'source_id': source_id,
        #             'experiment_id': experiment_id,
        #             'version': version,
        #             'table_id': table_id,
        #             'variable_id': variable_id,
        #             'time_range': None,
        #             'tier' : get_tier(file_path),
        #    }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

persiann_srs = filter_for_str(srs,'persiann')
persiann_df = split(persiann_srs, split_filename_PERSIANN)

persiann_df.head(5)

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PERSIANN-CDR,reanaly,v01r01,day,pr,201601-201612,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PERSIANN-CDR,reanaly,v01r01,day,pr,199801-199812,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PERSIANN-CDR,reanaly,v01r01,day,pr,201201-201212,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PERSIANN-CDR,reanaly,v01r01,day,pr,198501-198512,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PERSIANN-CDR,reanaly,v01r01,day,pr,200301-200312,Tier2


In [17]:
#  /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/PATMOS-x/OBS_PATMOS-x_sat_NOAA_Amon_clt_201201-201212.nc

def split_filename_PATMOS(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_PATMOS-x_sat_NOAA_Amon_clt_201201-201212.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{institution_id}_{table_id}_{variable_id}_{time_range}"


    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, institution_id, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'institution_id': institution_id,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

patmos_srs = filter_for_str(srs,'patmos')
patmos_df = split(patmos_srs, split_filename_PATMOS)
patmos_df = patmos_df.rename(columns={"version":"institution_id"})
patmos_df.head(5)

,path,project_id,source_id,experiment_id,institution_id,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PATMOS-x,sat,NOAA,Amon,clt,199501-199512,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PATMOS-x,sat,NOAA,Amon,clt,200901-200912,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PATMOS-x,sat,NOAA,Amon,clt,199001-199012,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PATMOS-x,sat,NOAA,Amon,clt,199701-199712,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,PATMOS-x,sat,NOAA,Amon,clt,199901-199912,Tier2


In [18]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/ISCCP-FH/OBS_ISCCP-FH_sat_v0_Amon_tas_198401-201612.nc

def split_filename_ISCCP(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_ISCCP-FH_sat_v0_Amon_tas_198401-201612.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"


    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

isccp_srs = filter_for_str(srs,'isccp-fh')
isccp_df = split(isccp_srs, split_filename_ISCCP)

isccp_df.head(5)

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ISCCP-FH,sat,v0,Amon,tas,198401-201612,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ISCCP-FH,sat,v0,Amon,rsus,198401-201612,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ISCCP-FH,sat,v0,Amon,rlus,198401-201612,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ISCCP-FH,sat,v0,Amon,rlutcs,198401-201612,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ISCCP-FH,sat,v0,Amon,prw,198401-201612,Tier2


In [19]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/E-OBS/OBS_E-OBS_ground_v20.0e-0.25_day_tas_195001-201907.nc

def split_filename_EOBS(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_E-OBS_ground_v20.0e-0.25_day_tas_195001-201907.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"


    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

eobs_srs = filter_for_str(srs,'e-obs')
eobs_df = split(eobs_srs, split_filename_EOBS)

eobs_df.head(5)

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,E-OBS,ground,v20.0e-0.25,day,tas,195001-201907,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,E-OBS,ground,v20.0e-0.25,day,psl,195001-201907,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,E-OBS,ground,v20.0e-0.25,Amon,tasmin,195001-201907,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,E-OBS,ground,v20.0e-0.25,Amon,tasmax,195001-201907,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,E-OBS,ground,v20.0e-0.25,day,tasmax,195001-201907,Tier2


In [20]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/ta_AIRS-2-1_BE_gn_200209-201609.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-0/hur_AIRS-2-0_L3_v2_200209-201105.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/hurNobs_mon_AIRS_obs4MIPs_V2.1_200209-201609.nc
def split_filename_AIRS(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `ta_AIRS-2-1_BE_gn_200209-201609` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"


    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id, experiment_id, grid_or_version, time_range]:
            if "v" in grid_or_version.lower():
                version = grid_or_version
            else:
                version = None
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': None,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [variable_id, table_id, source_id, project_id, grid_or_version, time_range]:
            if "v" in grid_or_version.lower():
                version = grid_or_version
            else:
                version = None
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': None,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

airs_srs = filter_for_str(srs,'airs')
airs_df = split(airs_srs, split_filename_AIRS)

airs_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS-2-0,L3,v2,None,hur,200209-201105,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS-2-1,BE,None,None,hus,200209-201609,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS-2-1,BE,None,None,hur,200209-201609,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS,None,V2.1,mon,hurNobs,200209-201609,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS-2-1,None,None,mon,hur,200209-201609,Tier1
5,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS,None,V2.1,mon,taStderr,200209-201609,Tier1
6,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS,None,V2.1,mon,taNobs,200209-201609,Tier1
7,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS,None,V2.1,mon,husNobs,200209-201609,Tier1
8,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS,None,V2.1,mon,hurStderr,200209-201609,Tier1
9,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,AIRS-2-1,None,None,mon,ta,200209-201609,Tier1


In [21]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/GPCP-SG/pr_GPCP-SG_L3_v2.2_197901-197912.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GPCP-SG/OBS_GPCP-SG_atmos_2.3_Amon_pr_197901-202304.nc
def split_filename_GPCP(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `pr_GPCP-SG_L3_v2.2_197901-197912.nc` which get mapped as
    f"{variable_id}_{source_id}_{experiment_id}_{version}_{time_range}"


    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [variable_id, source_id, experiment_id, version, time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': None,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

gpcp_srs = filter_for_str(srs,'gpcp-sg')
gpcp_df = split(gpcp_srs, split_filename_GPCP)

print(len(gpcp_df))
gpcp_df.head(5)

15


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GPCP-SG,atmos,2.3,Amon,pr,197901-202304,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,GPCP-SG,L3,v2.2,None,pr,197901-201312,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,GPCP-SG,L3,v2.3,None,pr,197901-201710,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,GPCP-SG,L3,v2.3,None,pr,197901-201710,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,GPCP-SG,L3,v2.2,None,prStderr,198001-198912,Tier1


In [22]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/Eppley-VGPM-MODIS/OBS_Eppley-VGPM-MODIS_sat_R2018_Omon_intpp_200207-201903.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/MODIS-1-0/clt_mon_MODIS-1-0_BE_gn_200003-201109.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/MODIS/cltNobs_MODIS_L3_C5_200003-201109.nc

def split_filename_MODIS(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `cltNobs_MODIS_L3_C5_200003-201109.nc` which get mapped as
    f"{variable_id}_{source_id}_{experiment_id}_{version}_{time_range}"
    or
    `clt_mon_MODIS-1-0_BE_gn_200003-201109.nc` which get mapped as
    f"{variable_id}_{table_id}_{source_id}_{experiment_id}_{grid_label}_{time_range}
    or 
    `OBS_Eppley-VGPM-MODIS_sat_R2018_Omon_intpp_200207-201903.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id, experiment_id, version, time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': None,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [variable_id, table_id, source_id, experiment_id, _, time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': None,
                    'table_id': None,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

modis_srs = filter_for_str(srs,'modis')
modis_df = split(modis_srs, split_filename_MODIS)

print(len(modis_df))
modis_df.head(5)

12


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Eppley-VGPM-MODIS,sat,R2018,Omon,intpp,200207-201903,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,MODIS,L3,C5,None,cltStddev,200003-201109,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,MODIS,L3,C5,None,od550aer,200003-201212,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,MODIS,L3,C5,None,od550aerStdv,200003-201212,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,MODIS,L3,C5,None,cltisccp,200003-201109,Tier1


In [23]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/CowtanWay/OBS_CowtanWay_reanaly_ghcn_short_krig_v2_Amon_tasa_197901-201908.nc

def split_filename_COWTAN(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_CowtanWay_reanaly_ghcn_short_krig_v2_Amon_tasa_197901-201908.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, *version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': '_'.join(version),
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

cowtan_srs = filter_for_str(srs,'cowtan')
cowtan_df = split(cowtan_srs, split_filename_COWTAN)

print(len(cowtan_df))
cowtan_df.head(5)

8


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CowtanWay,reanaly,had4_short_krig_v2,Amon,tasa,197901-201911,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CowtanWay,reanaly,had4sst4_krig_v2,Amon,tasa,185001-201812,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CowtanWay,reanaly,had4_short_uah_v2,Amon,tasa,197901-201707,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CowtanWay,reanaly,had4_krig_v2,Amon,tasa,185001-201911,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CowtanWay,reanaly,had4_krig_v1,Amon,tasa,197901-201312,Tier2


In [24]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/HadCRUT4/OBS_HadCRUT4_ground_1_Amon_tasa_185001-201812.nc

def split_filename_HADCRUT(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_HadCRUT4_ground_1_Amon_tasa_185001-201812.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

hadcrut_srs = filter_for_str(srs,'hadcrut')
hadcrut_df = split(hadcrut_srs, split_filename_HADCRUT)

print(len(hadcrut_df))
hadcrut_df.head(5)

11


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadCRUT5,ground,5.0.1.0-noninfilled,Amon,tasa,185001-202112,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadCRUT5,ground,5.0.1.0-analysis,Amon,tas,185001-202112,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadCRUT5,ground,analysis,Amon,tas,185001-202012,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadCRUT5,ground,noninfilled,Amon,tas,185001-202012,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadCRUT5,ground,5.0.1.0-analysis,Amon,tasa,185001-202112,Tier2


In [25]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/ATSR/tosAnom_ATSR_L3_ARC-v1.1.1_199701-201112.nc

def split_filename_ATSR(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `tosAnom_ATSR_L3_ARC-v1.1.1_199701-201112.nc` which get mapped as
    f"{variable_id}_{source_id}_{experiment_id}_{version}_{time_range}

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id, experiment_id, version, time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : None,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

atsr_srs = filter_for_str(srs,'atsr')
atsr_df = split(atsr_srs, split_filename_ATSR)

print(len(atsr_df))
atsr_df.head(5)

2


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ATSR,L3,ARC-v1.1.1,None,tos,199701-201112,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ATSR,L3,ARC-v1.1.1,None,tosAnom,199701-201112,Tier1


In [26]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/SSMI-MERIS/prw_SSMI-MERIS_L3_v1-00_200301-200812.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/SSMI/prw_SSMI_L3_RSSv07r00_198801_201512.nc


def split_filename_SSMI(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `prw_SSMI-MERIS_L3_v1-00_200301-200812.nc` which get mapped as
    f"{variable_id}_{source_id}_{experiment_id}_{version}_{time_range}
    or
    prw_SSMI_L3_RSSv07r00_198801_201512.nc

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id, experiment_id, version, *time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : None,
                    'variable_id': variable_id,
                    'time_range': '_'.join(time_range),
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

ssmi_srs = filter_for_str(srs,'ssmi')
ssmi_df = split(ssmi_srs, split_filename_SSMI)

print(len(ssmi_df))
ssmi_df.head(5)

5


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,SSMI,L3,RSSv07r00,None,prw,198801_201512,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,SSMI-MERIS,L3,v1-00,None,prw,200301-200812,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,SSMI-MERIS,L3,v1-00,None,prwFlag,200301-200812,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,SSMI-MERIS,L3,v1-00,None,prwStddev,200301-200812,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,SSMI-MERIS,L3,v1-00,None,prwErr,200301-200812,Tier1


In [27]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/TRMM-L3/prStderr_TRMM-L3_v7_7A_199801-201312.nc

def split_filename_TRMM(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `prStderr_TRMM-L3_v7_7A_199801-201312.nc` which get mapped as
    f"{variable_id}_{source_id}_*{version}_{time_range} 
    or
    `pr_TRMM-L3_v7-7A_199801-201312.nc` which get mapped as
    f"{variable_id}_{source_id}_{version}_{time_range} 

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id,  version, time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': None,
                    'version': version,
                    'table_id' : None,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [variable_id, source_id,  *version, time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': None,
                    'version': '-'.join(version),
                    'table_id' : None,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

trmm_srs = filter_for_str(srs,'trmm')
trmm_df = split(trmm_srs, split_filename_TRMM)

print(len(trmm_df))
trmm_df.head(5)

6


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,TRMM-L3,None,v7-7A,None,prStderr,199801-201312,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,TRMM-L3,None,v7-7A,None,prStderr,199801-201312,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,TRMM-L3,None,v7-7A,None,pr,199801-201312,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,TRMM-L3,None,v7-7A,None,pr,199801-201312,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,TRMM-L3,None,v7-7A,None,prStderr,199801-201312,Tier1


In [28]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/ghgcci/xch4_ghgcci_l3_v100_200301_201412.nc


def split_filename_GHGCCI(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `xch4_ghgcci_l3_v100_200301_201412.nc` which get mapped as
    f"{variable_id}_{source_id}_{experiment_id}_{version}_{*time_range} 

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id,experiment_id,  version, *time_range]:
            return {
                    'project_id': "obs4MIPs",
                    'source_id': source_id,
                    'experiment_id': None,
                    'version': version,
                    'table_id' : None,
                    'variable_id': variable_id,
                    'time_range': '-'.join(time_range),
                    'tier' : get_tier(file_path),
                }

        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

ghgcci_srs = filter_for_str(srs,'ghgcci')
ghgcci_df = split(ghgcci_srs, split_filename_GHGCCI)

print(len(ghgcci_df))
ghgcci_df.head(5)

4


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ghgcci,None,v100,None,xco2,200301-201412,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ghgcci,None,v100,None,xch4,200301-201412,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ghgcci,None,v100,None,xco2,200301-201412,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ghgcci,None,v100,None,xch4,200301-201412,Tier1


In [36]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/BerkeleyEarth/OBS_BerkeleyEarth_reanaly_2020_Amon_tas_185001-201912.nc


def split_filename_BERKELEY(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_BerkeleyEarth_reanaly_2020_Amon_tas_185001-201912.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    `OBS_BerkeleyEarth_reanaly_2020_fx_sftlf.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}"
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [project_id, source_id, experiment_id, version, table_id, variable_id]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': None,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

berkeley_srs = filter_for_str(srs,'berkeley')
berkeley_df = split(berkeley_srs, split_filename_BERKELEY)

print(len(berkeley_df))
berkeley_df.head(5)

3


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,BerkeleyEarth,reanaly,2020,Amon,tas,185001-201912,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,BerkeleyEarth,reanaly,2020,fx,sftlf,None,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,BerkeleyEarth,reanaly,2020,Amon,tasa,185001-201912,Tier2


In [44]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GPCC/OBS_GPCC_reanaly_v2018_10-numgauge1_Amon_pr_189101-201612.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GPCC/OBS_GPCC_reanaly_v2018_10_Amon_pr_189101-201612.nc

def split_filename_GPCC(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_GPCC_reanaly_v2018_10-numgauge1_Amon_pr_189101-201612.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{*version}_{table_id}_{variable_id}_{time_range} 
    or 
    `OBS_GPCC_reanaly_v2018_10_Amon_pr_189101-201612.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}"
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, *version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': '-'.join(version),
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

gpcc_srs = filter_for_str(srs,'/gpcc/')
gpcc_df = split(gpcc_srs, split_filename_GPCC)

print(len(gpcc_df))
gpcc_df.head(5)

8


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GPCC,reanaly,v2018-05-numgauge1,Amon,pr,189101-201612,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GPCC,reanaly,v2018-25,Amon,pr,189101-201612,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GPCC,reanaly,v2018-10-numgauge1,Amon,pr,189101-201612,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GPCC,reanaly,v2018-025-numgauge1,Amon,pr,189101-201612,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GPCC,reanaly,v2018-05,Amon,pr,189101-201612,Tier2


In [56]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GCP2018/OBS_GCP2018_reanaly_1.0_Omon_fgco2_195906-201706.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GCP2020/OBS_GCP2020_reanaly_1.0-residual_Lmon_nbp_195906-201906.nc

def split_filename_GCP20(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_GCP2018_reanaly_1.0_Omon_fgco2_195906-201706.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

gcp20_srs = filter_for_str(srs,'/gcp20')
gcp20_df = split(gcp20_srs, split_filename_GCP20)

print(len(gcp20_df))
gcp20_df.head(5)

5


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GCP2020,reanaly,1.0,Lmon,nbp,195906-201906,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GCP2020,reanaly,1.0-residual,Lmon,nbp,195906-201906,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GCP2020,reanaly,1.0,Omon,fgco2,195906-201906,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GCP2018,reanaly,1.0,Omon,fgco2,195906-201706,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GCP2018,reanaly,1.0,Lmon,nbp,195906-201706,Tier2


In [61]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GCP2018/OBS_GCP2018_reanaly_1.0_Omon_fgco2_195906-201706.nc
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GCP2020/OBS_GCP2020_reanaly_1.0-residual_Lmon_nbp_195906-201906.nc

def split_filename_CRU(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_CRU_reanaly_TS4.02_Amon_pr_190101-201712.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

cru_srs = filter_for_str(srs,'/cru/')
cru_df = split(cru_srs, split_filename_CRU)

print(len(cru_df))
cru_df

12


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CRU,reanaly,TS4.02,Amon,pr,190101-201712,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CRU,reanaly,TS4.02,Amon,tas,190101-201712,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,CRU,reanaly,TS4.07,Amon,tasmax,190101-202212,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CRU,reanaly,TS4.04,Amon,tas,190101-201912,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,CRU,reanaly,TS4.07,Amon,tasmin,190101-202212,Tier2
5,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CRU,reanaly,TS4.04,Amon,pr,190101-201912,Tier2
6,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,CRU,reanaly,TS4.07,Amon,tas,190101-202212,Tier2
7,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CRU,reanaly,TS4.04-stn1,Amon,tas,190101-201912,Tier2
8,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,CRU,reanaly,TS4.07,Amon,clt,190101-202212,Tier2
9,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,CRU,reanaly,TS4.07,Amon,pr,190101-202212,Tier2


In [65]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/HadISST/OBS_HadISST_reanaly_1_Amon_ts_187001-202112.nc

def split_filename_HADISST(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_HadISST_reanaly_1_Amon_ts_187001-202112.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

hadisst_srs = filter_for_str(srs,'hadisst')
hadisst_df = split(hadisst_srs, split_filename_HADISST)

print(len(hadisst_df))
hadisst_df

3


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadISST,reanaly,1,OImon,sic,187001-202112,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadISST,reanaly,1,Amon,ts,187001-202112,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HadISST,reanaly,1,Omon,tos,187001-202112,Tier2


In [72]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/Landschuetzer2016/OBS_Landschuetzer2016_clim_v2016_Omon_spco2_198201-201512.nc

def split_filename_LANDSCHUTZER(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_Landschuetzer2016_clim_v2016_Omon_spco2_198201-201512.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

landschutzer_srs = filter_for_str(srs,'landschuetzer')
landschutzer_df = split(landschutzer_srs, split_filename_LANDSCHUTZER)

print(len(landschutzer_df))
landschutzer_df

4


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,Landschuetzer2020,clim,1.1,Omon,spco2,200401-200412,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Landschuetzer2016,clim,v2016,Omon,spco2,198201-201512,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Landschuetzer2016,clim,v2016,Omon,fgco2,198201-201512,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Landschuetzer2016,clim,v2016,Omon,dpco2,198201-201512,Tier2


In [77]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/HALOE/OBS_HALOE_sat_1_Amon_hus_199101-200212.nc

def split_filename_HALOE(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_HALOE_sat_1_Amon_hus_199101-200212.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

haloe_srs = filter_for_str(srs,'haloe')
haloe_df = split(haloe_srs, split_filename_HALOE)

print(len(haloe_df))
haloe_df

2


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HALOE,sat,1,Amon,hus,199101-200212,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,HALOE,sat,1,Amon,tro3,199101-200212,Tier2


In [82]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GHCN/OBS_GHCN_ground_1_Amon_pr_190001-201412.nc

def split_filename_GHCN(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_GHCN_ground_1_Amon_pr_190001-201412.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

ghcn_srs = filter_for_str(srs,'/ghcn')
ghcn_df = split(ghcn_srs, split_filename_GHCN)

print(len(ghcn_df))
ghcn_df

2


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GHCN-CAMS,ground,1,Amon,tas,194801-202002,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GHCN,ground,1,Amon,pr,190001-201412,Tier2


In [85]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/GISTEMP/OBS_GISTEMP_ground_v4_Amon_tasa_188001-202001.nc

def split_filename_GISTEMP(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_GISTEMP_ground_v4_Amon_tasa_188001-202001.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

gistemp_srs = filter_for_str(srs,'gistemp')
gistemp_df = split(gistemp_srs, split_filename_GHCN)

print(len(gistemp_df))
gistemp_df

1


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GISTEMP,ground,v4,Amon,tasa,188001-202001,Tier2


In [95]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/Kadow2020/OBS_Kadow2020_ground_5.0.1.0_Amon_tasa_185001-202012.nc

def split_filename_KADOW(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_Kadow2020_ground_5.0.1.0_Amon_tasa_185001-202012.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

kadow_srs = filter_for_str(srs,'kadow')
kadow_df = split(kadow_srs, split_filename_KADOW)

print(len(kadow_df))
kadow_df

1


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Kadow2020,ground,5.0.1.0,Amon,tasa,185001-202012,Tier2


In [103]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/CALIPSO-GOCCP/OBS_CALIPSO-GOCCP_sat_3.1.2_cfMon_clcalipso_200701-201512.nc

def split_filename_CALIPSO(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_CALIPSO-GOCCP_sat_3.1.2_cfMon_clcalipso_200701-201512.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

calipso_srs = filter_for_str(srs,'/calipso-goccp/')
calipso_df = split(calipso_srs, split_filename_CALIPSO)

print(len(calipso_df))
calipso_df

1


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CALIPSO-GOCCP,sat,3.1.2,cfMon,clcalipso,200701-201512,Tier2


In [113]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/Duveiller2018/OBS_Duveiller2018_clim_v2018_Amon_albDiffiTr13_201001-201012.nc

def split_filename_DUVEILLER(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_Duveiller2018_clim_v2018_Amon_albDiffiTr13_201001-201012.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

duveiller_srs = filter_for_str(srs,'duveiller2018')
duveiller_df = split(duveiller_srs, split_filename_DUVEILLER)

print(len(duveiller_df))
duveiller_df

1


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Duveiller2018,clim,v2018,Amon,albDiffiTr13,201001-201012,Tier2


In [107]:
# /g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/NOAAGlobalTemp/OBS_NOAAGlobalTemp_ground_v5.0.0_Amon_tasa_188001-202205.nc

def split_filename_NOAAGBLTMP(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_NOAAGlobalTemp_ground_v5.0.0_Amon_tasa_188001-202205.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range} 
    or 
    

    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id' : table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

noaagbltmp_srs = filter_for_str(srs,'/noaaglobaltemp/')
noaagbltmp_df = split(noaagbltmp_srs, split_filename_NOAAGBLTMP)

print(len(noaagbltmp_df))
noaagbltmp_df

1


,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NOAAGlobalTemp,ground,v5.0.0,Amon,tasa,188001-202205,Tier2


In [138]:
merged_df_1 = pd.concat(
    [
        obs6_df, obs4mips_df, cfsr_df, ceres_ebaf_df, jra55_df, esacci_df, esrl_df, ncep_df, wfde_df, osi450_df, regen_df,
        woa_df, piomas_df, persiann_df, patmos_df, isccp_df, eobs_df, airs_df, gpcp_df, modis_df, cowtan_df, hadcrut_df,
        atsr_df, ssmi_df, trmm_df, ghgcci_df, berkeley_df, gpcc_df, gcp20_df, cru_df, hadisst_df, landschutzer_df, 
        haloe_df, ghcn_df, gistemp_df, kadow_df, calipso_df, duveiller_df, noaagbltmp_df
    ]
).reset_index(drop=True).drop_duplicates()

# Now we're going to drop on path, keeping the last column. TLDR; is the duplicates are cases that are handled slightly 
# better by the later functions matching on attributes


merged_df_1.drop_duplicates(subset='path', keep='last', inplace=True)

In [115]:
files_to_do = set(srs) - set(merged_df_1.path)
sorted(files_to_do)

[]

In [140]:
merged_df_1

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier,institution_id
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,ERA-Interim,reanaly,1,Emon,tdps,201701-201712,Tier3,NaN
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,ERA-Interim,reanaly,1,day,rsds,201101-201112,Tier3,NaN
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,ERA-Interim,reanaly,1,Amon,hur,199401-199412,Tier3,NaN
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,ERA-Interim,reanaly,1,Amon,rsutcs,199401-199412,Tier3,NaN
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,ERA-Interim,reanaly,1,Amon,vas,200801-200812,Tier3,NaN
...,...,...,...,...,...,...,...,...,...,...
7617,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,GISTEMP,ground,v4,Amon,tasa,188001-202001,Tier2,NaN
7618,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Kadow2020,ground,5.0.1.0,Amon,tasa,185001-202012,Tier2,NaN
7619,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CALIPSO-GOCCP,sat,3.1.2,cfMon,clcalipso,200701-201512,Tier2,NaN
7620,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,Duveiller2018,clim,v2018,Amon,albDiffiTr13,201001-201012,Tier2,NaN


In [144]:
!pwd

/home/189/ct1163


In [158]:
merged_df_1.to_csv("/home/189/ct1163/ESMValTool-workflow/observations/catalog/esm/access-ct11.csv.gz")

In [149]:
import json

In [150]:
data = {
   "esmcat_version": "0.1.0",
   "attributes": [
   	  {
   	      "column_name": "project_id"
   	  },
   	  {
   	      "column_name": "source_id"
   	  },
   	  {
   	      "column_name": "experiment_id"
   	  },
   	  {
   	      "column_name": "version"
   	  },
   	  {
   	      "column_name": "table_id"
   	  },
   	  {
   	      "column_name": "variable_id"
   	  },
   	  {
   	      "column_name": "time_range"
   	  },
   	  {
   	      "column_name": "tier"
   	  },
   	  {
   	      "column_name": "institution_id"
   	  }
   ],
   "assets": {
  	"column_name": "path",
   	"format": "netcdf"
   },
   "aggregation_control": {
  	"variable_column_name": "variable_id",
   	"groupby_attrs": [
   		"project_id",
   		"source_id",
   		"experiment_id",
   		"version",
   		"table_id",
   		"variable_id",
		"time_range",
   		"tier",
   		"institution_id"
	],
   	"aggregations": [
   	       {
   	           "type": "join_existing",
   		    "attribute_name": "time_range",
   		    "options": {
			"dim": "time"
		   }
   	       }
   	]
   },
   "id": "access-nri-ct11",
   "title": "access-nri-ct11",
   "description": "ACCESS-NRI Replica datasets for model evaluation. Includes ESMValTool data pool",
  "catalog_file": "file:///g/data/xp65/public/apps/access-nri-intake-catalog/ct11_tmpdir_catalog/access-ct11.csv.gz"
  }

In [159]:
with open('/home/189/ct1163/ESMValTool-workflow/observations/catalog/esm/catalog.json', 'w') as json_file:
    json.dump(data, json_file)
